In [3]:
import numpy as np
import scipy.special as math_gamma

N, M, L_k, Pa, MaxIter, alpha, beta = 15, 3, 5, 0.25, 50, 0.01, 1.5
D = M * L_k
T = np.random.uniform(10, 45, size=(D, D))
np.fill_diagonal(T, 0)

def evaluate_cost(x):
    routes = np.reshape(np.clip(np.floor(x).astype(int), 0, D - 1), (M, L_k))
    total_time = sum(T[r[i], r[i+1]] for r in routes for i in range(L_k - 1))
    return total_time + 50.0 * max(0, total_time - (D * 15))

def levy_step():
    sigma = ((math_gamma.gamma(1 + beta) * np.sin(np.pi * beta / 2)) / 
             (math_gamma.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))) ** (1 / beta)
    return np.random.normal(0, sigma, D) / (np.abs(np.random.normal(0, 1, D)) ** (1 / beta))

nests = np.random.uniform(0, D, size=(N, D))
fitness = np.array([evaluate_cost(x) for x in nests])
x_best, f_best = nests[np.argmin(fitness)].copy(), np.min(fitness)

for t in range(MaxIter):
    for i in range(N):
        x_new = np.clip(nests[i] + alpha * levy_step(), 0, D - 1)
        f_new = evaluate_cost(x_new)
        j = np.random.randint(0, N)
        if f_new < fitness[j]:
            nests[j], fitness[j] = x_new, f_new

    for i in range(N):
        if np.random.rand() < Pa:
            j, k = np.random.choice(N, size=2, replace=False)
            x_new = np.clip(nests[i] + np.random.rand() * (nests[j] - nests[k]), 0, D - 1)
            f_new = evaluate_cost(x_new)
            if f_new < fitness[i]:
                nests[i], fitness[i] = x_new, f_new

    if np.min(fitness) < f_best:
        x_best, f_best = nests[np.argmin(fitness)].copy(), np.min(fitness)
        
print(f"Optimal Cost: {f_best:.2f}")

Optimal Cost: 221.00
